# Chapter 10 — Sorting and Searching

Know these cold:

| Algorithm | Average | Worst | Memory |
|---|---|---|---|
| Bubble / Selection | O(n²) | O(n²) | O(1) |
| Merge sort | O(n log n) | O(n log n) | O(n) |
| Quick sort | O(n log n) | O(n²) | O(log n) |
| Radix sort | O(kn) | O(kn) | O(n + k) |

Python's `sorted`/`list.sort` is **Timsort**, a stable hybrid of merge and insertion sort. `bisect` gives binary search.

In [1]:
import bisect
import random
from collections import defaultdict


def bubble_sort(a: list[int]) -> list[int]:
    a = a[:]
    for end in range(len(a) - 1, 0, -1):
        swapped = False
        for i in range(end):
            if a[i] > a[i + 1]:
                a[i], a[i + 1] = a[i + 1], a[i]
                swapped = True
        if not swapped:
            break
    return a


def selection_sort(a: list[int]) -> list[int]:
    a = a[:]
    for i in range(len(a)):
        m = min(range(i, len(a)), key=a.__getitem__)
        a[i], a[m] = a[m], a[i]
    return a


def merge_sort(a: list[int]) -> list[int]:
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    left, right = merge_sort(a[:mid]), merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    return out + left[i:] + right[j:]


def quick_sort(a: list[int], lo: int = 0, hi: int | None = None) -> list[int]:
    if hi is None:
        a = a[:]
        hi = len(a) - 1
    if lo < hi:
        pivot = a[(lo + hi) // 2]
        i, j = lo, hi
        while i <= j:
            while a[i] < pivot: i += 1
            while a[j] > pivot: j -= 1
            if i <= j:
                a[i], a[j] = a[j], a[i]
                i += 1; j -= 1
        quick_sort(a, lo, j)
        quick_sort(a, i, hi)
    return a


def radix_sort(a: list[int]) -> list[int]:
    """LSD radix sort for non-negative ints, base 10."""
    if not a:
        return a
    exp, out = 1, a[:]
    while max(a) // exp:
        buckets: list[list[int]] = [[] for _ in range(10)]
        for x in out:
            buckets[x // exp % 10].append(x)
        out = [x for b in buckets for x in b]
        exp *= 10
    return out


def binary_search(a: list[int], x: int) -> int:
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if a[mid] < x: lo = mid + 1
        elif a[mid] > x: hi = mid - 1
        else: return mid
    return -1


data = [random.randint(0, 999) for _ in range(500)]
for sorter in (bubble_sort, selection_sort, merge_sort, quick_sort, radix_sort):
    assert sorter(data) == sorted(data), sorter.__name__
s = sorted(set(data))
assert all(binary_search(s, x) == s.index(x) for x in s) and binary_search(s, -5) == -1
print("sorting & searching basics ok")

sorting & searching basics ok


## 10.1 Sorted Merge
A and B are sorted, and A has enough empty space at the end to hold B. Merge B into A in place. Fill from the **back** so nothing gets overwritten.

In [2]:
def sorted_merge(a: list[int], b: list[int], count_a: int) -> list[int]:
    i, j, k = count_a - 1, len(b) - 1, count_a + len(b) - 1
    while j >= 0:
        if i >= 0 and a[i] > b[j]:
            a[k] = a[i]; i -= 1
        else:
            a[k] = b[j]; j -= 1
        k -= 1
    return a


A = [1, 3, 5, 7, None, None, None]
assert sorted_merge(A, [2, 4, 8], 4) == [1, 2, 3, 4, 5, 7, 8]
print("10.1 ok")

10.1 ok


## 10.2 Group Anagrams
Sort an array of strings so anagrams end up next to each other. Group by a canonical key (the sorted letters) in a hash map: `O(n · k log k)`.

In [3]:
def group_anagrams(words: list[str]) -> list[str]:
    groups: dict[str, list[str]] = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)
    return [w for g in groups.values() for w in g]


out = group_anagrams(["acre", "dog", "care", "god", "race", "cat"])
assert out == ["acre", "care", "race", "dog", "god", "cat"]
print("10.2 ok")

10.2 ok


## 10.3 Search in Rotated Array
A sorted array has been rotated an unknown number of times. Find an element. Modified binary search: one half is always normally ordered. With duplicates (`left == mid`), we may have to search both halves, which degrades to `O(n)` worst case.

In [4]:
def search_rotated(a: list[int], x: int, lo: int = 0, hi: int | None = None) -> int:
    if hi is None:
        hi = len(a) - 1
    if lo > hi:
        return -1
    mid = (lo + hi) // 2
    if a[mid] == x:
        return mid
    if a[lo] < a[mid]:  # left half ordered
        if a[lo] <= x < a[mid]:
            return search_rotated(a, x, lo, mid - 1)
        return search_rotated(a, x, mid + 1, hi)
    if a[mid] < a[lo]:  # right half ordered
        if a[mid] < x <= a[hi]:
            return search_rotated(a, x, mid + 1, hi)
        return search_rotated(a, x, lo, mid - 1)
    # a[lo] == a[mid]: left half is all repeats, or we can't tell
    if a[mid] != a[hi]:
        return search_rotated(a, x, mid + 1, hi)
    r = search_rotated(a, x, lo, mid - 1)
    return r if r != -1 else search_rotated(a, x, mid + 1, hi)


arr = [15, 16, 19, 20, 25, 1, 3, 4, 5, 7, 10, 14]
assert search_rotated(arr, 5) == 8 and search_rotated(arr, 6) == -1
dup = [2, 2, 2, 3, 4, 2]
assert dup[search_rotated(dup, 4)] == 4 and search_rotated(dup, 5) == -1
for _ in range(200):
    base = sorted(random.sample(range(100), 20))
    k = random.randrange(20)
    rot = base[k:] + base[:k]
    x = random.choice(rot)
    assert rot[search_rotated(rot, x)] == x
print("10.3 ok")

10.3 ok


## 10.4 Sorted Search, No Size
`Listy` is sorted, has no `size()`, and `element_at(i)` returns -1 when out of bounds. Find the index of x. Grow the bound exponentially, then binary-search it (treating -1 as "too big"). `O(log n)`.

In [5]:
class Listy:
    def __init__(self, xs: list[int]) -> None:
        self._xs = xs

    def element_at(self, i: int) -> int:
        return self._xs[i] if 0 <= i < len(self._xs) else -1


def listy_search(lst: Listy, x: int) -> int:
    index = 1
    while lst.element_at(index) != -1 and lst.element_at(index) < x:
        index *= 2
    lo, hi = index // 2, index
    while lo <= hi:
        mid = (lo + hi) // 2
        v = lst.element_at(mid)
        if v == x:
            return mid
        if v > x or v == -1:
            hi = mid - 1
        else:
            lo = mid + 1
    return -1


L = Listy(list(range(0, 3000, 3)))
assert listy_search(L, 999) == 333 and listy_search(L, 1000) == -1 and listy_search(L, 0) == 0
print("10.4 ok")

10.4 ok


## 10.5 Sparse Search
A sorted array of strings is sprinkled with empty strings. Find a given string. On landing on an empty string, move to the nearest non-empty one, then continue the binary search. Worst case is `O(n)` if nearly everything is empty.

In [6]:
def sparse_search(a: list[str], x: str) -> int:
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if not a[mid]:
            left, right = mid - 1, mid + 1
            while True:
                if left < lo and right > hi:
                    return -1
                if right <= hi and a[right]:
                    mid = right; break
                if left >= lo and a[left]:
                    mid = left; break
                left -= 1; right += 1
        if a[mid] == x:
            return mid
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1


sp = ["at", "", "", "", "ball", "", "", "car", "", "", "dad", "", ""]
assert sparse_search(sp, "ball") == 4 and sparse_search(sp, "ta") == -1
assert all(sparse_search(sp, w) == i for i, w in enumerate(sp) if w)
print("10.5 ok")

10.5 ok


## 10.6 Sort Big File
Sort a 20 GB file with one string per line. It won't fit in memory, so use an **external sort**. Read chunks that fit in memory, sort each one, write it out as a "run", then do a k-way merge of the runs with a heap (`heapq.merge`). Below, the chunks are temp files.

In [7]:
import heapq
import os
import tempfile


def external_sort(lines, chunk_size: int) -> list[str]:
    runs: list[str] = []
    tmpdir = tempfile.mkdtemp()
    chunk: list[str] = []

    def flush() -> None:
        path = os.path.join(tmpdir, f"run{len(runs)}.txt")
        with open(path, "w") as f:
            f.writelines(x + "\n" for x in sorted(chunk))
        runs.append(path)
        chunk.clear()

    for line in lines:
        chunk.append(line)
        if len(chunk) == chunk_size:
            flush()
    if chunk:
        flush()
    files = [open(p) for p in runs]
    try:
        return [x.rstrip("\n") for x in heapq.merge(*files)]
    finally:
        for f in files:
            f.close()


words = ["".join(random.choices("abcdefg", k=5)) for _ in range(5000)]
assert external_sort(words, chunk_size=300) == sorted(words)
print("10.6 ok")

10.6 ok


## 10.7 Missing Int
Given 4 billion non-negative ints and 1 GB of memory, find an int that isn't in the file. A bit vector of 2³² bits is 512 MB, which fits.
Follow-up with only 10 MB: make two passes. The first pass counts how many values fall in each block of size `2^k`. Pick a block that isn't full, then use a bit vector over just that block. We scale the problem down to demo it.

In [8]:
def missing_int_bitvector(nums, universe: int) -> int:
    bits = bytearray(universe // 8 + 1)
    for n in nums:
        bits[n >> 3] |= 1 << (n & 7)
    for i in range(universe):
        if not bits[i >> 3] & (1 << (i & 7)):
            return i
    return -1


def missing_int_two_pass(get_nums, universe: int, block: int) -> int:
    counts = [0] * (universe // block + 1)
    for n in get_nums():                           # pass 1
        counts[n // block] += 1
    b = next(i for i, c in enumerate(counts) if c < block)
    bits = bytearray(block // 8 + 1)
    for n in get_nums():                           # pass 2
        if b * block <= n < (b + 1) * block:
            off = n - b * block
            bits[off >> 3] |= 1 << (off & 7)
    return b * block + next(i for i in range(block) if not bits[i >> 3] & (1 << (i & 7)))


U = 1 << 20
missing = 777_777
nums = [i for i in range(U) if i != missing]
random.shuffle(nums)
assert missing_int_bitvector(nums, U) == missing
assert missing_int_two_pass(lambda: iter(nums), U, block=1 << 10) == missing
print("10.7 ok")

10.7 ok


## 10.8 Find Duplicates
An array holds numbers from 1 to N (N ≤ 32,000) with duplicates, and only 4 KB of memory is available. Print every duplicate. 4 KB = 32,768 bits is enough for a bit vector over the range.

In [9]:
def find_duplicates(nums: list[int]) -> list[int]:
    bits = bytearray(4 * 1024)  # exactly 4 KB
    dups = []
    for n in nums:
        i = n - 1
        if bits[i >> 3] & (1 << (i & 7)):
            dups.append(n)
        else:
            bits[i >> 3] |= 1 << (i & 7)
    return dups


assert find_duplicates([1, 5, 32000, 5, 7, 1, 32000]) == [5, 1, 32000]
print("10.8 ok")

10.8 ok


## 10.9 Sorted Matrix Search
Every row and every column of an M×N matrix is sorted ascending. Find an element. Start in the top-right corner and step left if the value is too big, down if it's too small: `O(M + N)`.

In [10]:
def sorted_matrix_search(m: list[list[int]], x: int) -> tuple[int, int] | None:
    r, c = 0, len(m[0]) - 1
    while r < len(m) and c >= 0:
        if m[r][c] == x:
            return r, c
        if m[r][c] > x:
            c -= 1
        else:
            r += 1
    return None


M = [[15, 20, 40, 85], [20, 35, 80, 95], [30, 55, 95, 105], [40, 80, 100, 120]]
assert sorted_matrix_search(M, 55) == (2, 1) and sorted_matrix_search(M, 56) is None
print("10.9 ok")

10.9 ok


## 10.10 Rank from Stream
Numbers arrive in a stream. `track(x)` records one, and `get_rank_of_number(x)` returns how many values ≤ x have been seen (excluding x itself). Use a BST where each node stores its left-subtree size: `O(log n)` on average.

In [11]:
class RankNode:
    __slots__ = ("val", "left", "right", "left_size")

    def __init__(self, val: int) -> None:
        self.val, self.left, self.right, self.left_size = val, None, None, 0

    def insert(self, x: int) -> None:
        if x <= self.val:
            self.left_size += 1
            if self.left: self.left.insert(x)
            else: self.left = RankNode(x)
        else:
            if self.right: self.right.insert(x)
            else: self.right = RankNode(x)

    def rank(self, x: int) -> int:
        if x == self.val:
            return self.left_size
        if x < self.val:
            return self.left.rank(x) if self.left else -1
        r = self.right.rank(x) if self.right else -1
        return -1 if r == -1 else self.left_size + 1 + r


class StreamRank:
    def __init__(self) -> None:
        self.root: RankNode | None = None

    def track(self, x: int) -> None:
        if self.root: self.root.insert(x)
        else: self.root = RankNode(x)

    def get_rank_of_number(self, x: int) -> int:
        return self.root.rank(x) if self.root else -1


sr = StreamRank()
for v in [5, 1, 4, 4, 5, 9, 7, 13, 3]:
    sr.track(v)
assert [sr.get_rank_of_number(x) for x in (1, 3, 4)] == [0, 1, 3]
print("10.10 ok")

10.10 ok


## 10.11 Peaks and Valleys
Rearrange an array into alternating peaks and valleys (a peak is ≥ its neighbours, a valley ≤ them). `O(n)`: for each odd index, swap the largest of the three elements around it into that spot, so every odd index becomes a peak.

In [12]:
def peaks_and_valleys(a: list[int]) -> list[int]:
    for i in range(1, len(a), 2):
        biggest = max((j for j in (i - 1, i, i + 1) if j < len(a)), key=a.__getitem__)
        a[i], a[biggest] = a[biggest], a[i]
    return a


def is_peak_valley(a: list[int]) -> bool:
    return all((a[i] >= a[i - 1] and (i + 1 >= len(a) or a[i] >= a[i + 1])) for i in range(1, len(a), 2))


assert is_peak_valley(peaks_and_valleys([5, 3, 1, 2, 3]))
for _ in range(500):
    xs = [random.randint(0, 9) for _ in range(random.randint(0, 12))]
    assert is_peak_valley(peaks_and_valleys(xs))
print("10.11 ok")

10.11 ok
